In [2]:
import torch

from tqdm import tqdm
import yaml
from pathlib import Path

from utils import (
    Config, load_checkpoint,  set_seed,
)
from models import get_model
from optimizers import get_optimizer
from schedulers import get_scheduler
from dataset import get_dataloader
from losses import get_loss

In [3]:
CONFIG_PATH = Path("configs/baseline_mae_experiment.yaml")
CHECKPOINT_PATH = Path("checkpoints/baseline_mae_experiment_checkpoint.pt")

In [6]:
if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config path not found: {CONFIG_PATH}")

if CHECKPOINT_PATH and not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"Checkpoint path not found: {CHECKPOINT_PATH}") 

# config
with open(CONFIG_PATH, "r") as f:
    config_dict = yaml.safe_load(f)
config = Config.from_dict(config_dict)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
set_seed(config.seed)

# data
loader = get_dataloader(config, split="valid")

# model / optimizer / scheduler / loss
model = get_model(
    config,
    enc_in_size=loader.dataset.enc_dim,
    dec_in_size=loader.dataset.dec_dim,
    horizon=loader.dataset.horizon,
).to(device)
optimizer = get_optimizer(config, model.parameters())
scheduler = get_scheduler(config, optimizer)
criterion = get_loss(config)

# load checkpoint
curr_epoch = 0
train_losses = []
val_losses = []

model, state_dict = load_checkpoint(
    path=CHECKPOINT_PATH, 
    model=model,
    map_location=device,
)
optimizer.load_state_dict(state_dict["optim"])

curr_epoch = state_dict["epoch"] + 1
train_losses = state_dict["train_loss"]
val_losses = state_dict["val_loss"]
print(f"Resumed '{CHECKPOINT_PATH}' at epoch {curr_epoch}.")

[valid] Dataloader initialized with: num_workers=3, batch_size=512.
Model: LSTM
Optimizer: AdamW
LR Scheduler: CosineAnnealingLR
Loss function: MAE
Resumed 'checkpoints\baseline_mae_experiment_checkpoint.pt' at epoch 20.


In [12]:
print(f"Last train loss: {train_losses[-1]:.3f}\nLast validation loss: {val_losses[-1]:.3f}")

Last train loss: 0.536
Last validation loss: 0.576
